<a href="https://colab.research.google.com/github/AntonRize/WILL/blob/main/Colab_Notebooks/WILL_RG_Clean_Sample_Verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WILL RG Galactic Ladder: Clean-Sample Verification (full reproduction)

This notebook reproduces every number in the section **Clean-Sample Verification: Methodology and Results** of *WILL RG Galactic Dynamics*.

It reproduces, in order:

1. The global scale $a_\kappa$ and $3\pi R_H$, derived from $T_0$, $\alpha$, $G$, $\sigma_{SB}$ and $c$.
2. The sequential cleaning table (criteria C1 to C6), ending with 31 galaxies.
3. The table of surviving galaxies by origin of the catalogue distance.
4. The three result tables: red-giant tip and Cepheids, Ursa Major, and Hubble flow.
5. The summary table by group.
6. The diagnostics of the galaxies rejected by criterion C3.
7. The check that $D_{RG}$ contains no trace of the catalogue distance.

The last cell compares every reproduced number with the value printed in the paper and reports PASS or FAIL for each one.

No quantity is fitted, and no quantity is adjusted per galaxy. The only inputs are the SPARC data files and the physical constants listed below.

## 1. The global scale

The only global input is the structural resonance scale. It is computed from the CMB temperature and the fine-structure constant through the chain of the section *Deriving $H_0$ from CMB Temperature and $\alpha$*:

$$\rho_\gamma=\frac{4\sigma_{SB}T_0^4}{c^3},\qquad \rho_{max}=\frac{\rho_\gamma}{3\alpha^2},\qquad H_0=\sqrt{8\pi G\rho_{max}},\qquad a_\kappa=\frac{cH_0}{3\pi},\qquad 3\pi R_H=\frac{c^2}{a_\kappa}.$$

The conversion to km/s/Mpc uses $1\ \mathrm{Mpc}=3.0857\times10^{22}$ m, as in the paper.

In [1]:
import numpy as np

# Physical constants (CODATA values used in the paper)
c       = 299792458.0        # speed of light [m/s]
G       = 6.674e-11          # gravitational constant [m^3 kg^-1 s^-2]
sigma_SB= 5.67037e-8         # Stefan-Boltzmann constant [W m^-2 K^-4]
T0      = 2.7255             # CMB monopole temperature [K]
alpha   = 7.29735e-3         # fine-structure constant
MPC_M   = 3.0857e22          # metres per megaparsec (paper convention)

rho_gamma = 4.0 * sigma_SB * T0**4 / c**3        # radiation mass density [kg/m^3]
rho_max   = rho_gamma / (3.0 * alpha**2)         # saturation density [kg/m^3]
H0_SI     = np.sqrt(8.0 * np.pi * G * rho_max)   # [1/s]
H0        = H0_SI * MPC_M / 1000.0               # [km/s/Mpc]
a_kappa   = c * H0_SI / (3.0 * np.pi)            # structural resonance scale [m/s^2]
THREE_PI_RH_MPC = c**2 / a_kappa / MPC_M         # 3*pi*R_H [Mpc]

print(f"rho_gamma = {rho_gamma:.5e} kg/m^3")
print(f"rho_max   = {rho_max:.5e} kg/m^3")
print(f"H0        = {H0:.4f} km/s/Mpc")
print(f"a_kappa   = {a_kappa:.4e} m/s^2")
print(f"3 pi R_H  = {THREE_PI_RH_MPC:.2f} Mpc")

rho_gamma = 4.64509e-31 kg/m^3
rho_max   = 2.90765e-27 kg/m^3
H0        = 68.1456 km/s/Mpc
a_kappa   = 7.0248e-11 m/s^2
3 pi R_H  = 41462.35 Mpc


## 2. The data

All data come from the public SPARC database (Lelli, McGaugh & Schombert 2016, AJ 152, 157):

* `SPARC_Lelli2016c.mrt` is the galaxy table. It gives the catalogue distance $D_{SPARC}$, its error $e_D$, the distance method `f_D`, the inclination, and the quality flag `Q`.
* `Rotmod_LTG.zip` holds one mass-model file per galaxy. Each row gives the radius `Rad` [kpc], the observed speed $V_{obs}$ and its error, and the baryonic speeds $V_{gas}$, $V_{disk}$ and $V_{bul}$ [km/s].

The cell downloads both files from the official SPARC site. If that site is unreachable, it falls back to public GitHub copies. It prints the SHA-256 hash of every downloaded file, so that anyone can confirm they used the same data.

The distance-method codes are: 1 = Hubble flow ($H_0=73$), 2 = tip of the red giant branch, 3 = Cepheids, 4 = Ursa Major cluster, 5 = supernovae.

In [2]:
import io, zipfile, hashlib, urllib.request

def fetch(urls):
    # Try each URL in turn and return (bytes, url) for the first that works.
    for u in urls:
        try:
            req = urllib.request.Request(u, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as r:
                b = r.read()
            if len(b) > 100:
                return b, u
        except Exception as e:
            print(f"  could not read {u}: {e}")
    raise RuntimeError("No source reachable for " + urls[0])

TABLE1_URLS = ["https://astroweb.cwru.edu/SPARC/SPARC_Lelli2016c.mrt",
               "https://raw.githubusercontent.com/EntropyWizardchaos/sparc-coherence-test/master/SPARC_Lelli2016c.mrt"]
ROTMOD_ZIP_URL = "https://astroweb.cwru.edu/SPARC/Rotmod_LTG.zip"
ROTMOD_MIRROR  = "https://raw.githubusercontent.com/carsondowns-cte/Rotmod_LTG/main/{name}_rotmod.dat"

# Galaxy table
t1_bytes, t1_src = fetch(TABLE1_URLS)
print("Galaxy table from:", t1_src)
print("  SHA-256:", hashlib.sha256(t1_bytes).hexdigest())

lines = t1_bytes.decode("latin-1").replace("\r", "").split("\n")
last_rule = max(i for i, l in enumerate(lines) if l.startswith("-----"))
catalogue = {}
for l in lines[last_rule + 1:]:
    t = l.split()
    if not t:
        continue
    catalogue[t[0]] = dict(D=float(t[2]), eD=float(t[3]), fD=int(t[4]),
                           inc=float(t[5]), Q=int(t[17]))
print("Galaxies in table:", len(catalogue))

# Mass-model files
def parse_rotmod(text):
    rows = [[float(x) for x in l.split()[:6]] for l in text.replace("\r", "").split("\n")
            if l.strip() and not l.startswith("#")]
    return np.array(rows)

rotmod = {}
try:
    zb, zsrc = fetch([ROTMOD_ZIP_URL])
    print("Mass models from:", zsrc)
    print("  SHA-256:", hashlib.sha256(zb).hexdigest())
    with zipfile.ZipFile(io.BytesIO(zb)) as z:
        for n in z.namelist():
            if n.endswith("_rotmod.dat"):
                name = n.split("/")[-1].replace("_rotmod.dat", "")
                rotmod[name] = parse_rotmod(z.read(n).decode("latin-1"))
except RuntimeError:
    print("Official zip unreachable; using the GitHub mirror file by file.")
    h = hashlib.sha256()
    for name in sorted(catalogue):
        b, _ = fetch([ROTMOD_MIRROR.format(name=name)])
        h.update(b)
        rotmod[name] = parse_rotmod(b.decode("latin-1"))
    print("  combined SHA-256 of all mirror files:", h.hexdigest())
print("Mass-model files read:", len(rotmod))

Galaxy table from: https://astroweb.cwru.edu/SPARC/SPARC_Lelli2016c.mrt
  SHA-256: 5aa0501f6b0d881fa579030e315e7b5b6ef561a5bd3a07472f9929c7e5728243
Galaxies in table: 175
Mass models from: https://astroweb.cwru.edu/SPARC/Rotmod_LTG.zip
  SHA-256: 0a80cc90714828cc28b7dd57923576714d209f2490328c087c4a4ad607faf588
Mass-model files read: 175


## 3. The quantities computed at every point

For each tabulated point $i$ of a rotation curve ($i=1$ is the innermost):

**Angular radius.** The catalogue radius is divided by the catalogue distance. `Rad` is in kpc and $D_{SPARC}$ in Mpc, so the factor 1000 converts the ratio to radians:
$$\theta_i=\frac{Rad_i}{1000\,D_{SPARC}}.$$

**Kinematic projection.** The Doppler speed divided by the speed of light:
$$\beta_{obs,i}=\frac{V_{obs,i}}{c}.$$

**Baryonic speed.** The gas term keeps its sign, because a negative $V_{gas}$ is the outward pull of a gas ring lying outside the point. The stellar disk uses $\Upsilon_{disk}=0.5$ and the bulge $\Upsilon_{bulge}=0.7$:
$$V_{bar,i}^2=\operatorname{sign}(V_{gas,i})\,V_{gas,i}^2+0.5\,V_{disk,i}^2+0.7\,V_{bul,i}^2.$$

**Baryonic acceleration.** The squared baryonic speed divided by the radius:
$$g_{bar,i}=\frac{V_{bar,i}^2}{Rad_i}.$$

**Fundamental Tone eccentricity.** The ratio of the structural scale to the baryonic acceleration:
$$e_{0,i}=\sqrt{\frac{a_\kappa}{g_{bar,i}}}.$$

**Galactic Overtone distance at this point:**
$$D(\theta_i)=\frac{3\pi R_H\,\beta_{obs,i}^2}{\theta_i}\cdot\frac{e_{0,i}^2}{1+e_{0,i}}.$$

In [3]:
KPC_M = MPC_M / 1000.0     # metres per kiloparsec
C_KMS = c / 1000.0         # speed of light [km/s]
UPS_DISK, UPS_BULGE = 0.5, 0.7

def point_quantities(name, D_cat=None, scale=1.0):
    # scale multiplies the catalogue distance and every column proportional to it
    # (Rad, Vgas^2, Vdisk^2, Vbul^2); it is used only in the independence check.
    d = rotmod[name]
    Rad  = d[:, 0] * scale
    Vobs, eV = d[:, 1], d[:, 2]
    Vgas, Vdisk, Vbul = d[:, 3] * np.sqrt(scale), d[:, 4] * np.sqrt(scale), d[:, 5] * np.sqrt(scale)
    D = (catalogue[name]["D"] if D_cat is None else D_cat) * scale

    gas_term  = np.sign(Vgas) * Vgas**2                 # (km/s)^2, signed
    disk_term = UPS_DISK * Vdisk**2
    bul_term  = UPS_BULGE * Vbul**2
    Vbar2 = gas_term + disk_term + bul_term             # (km/s)^2

    theta = Rad / (1000.0 * D)                          # radians
    beta  = Vobs / C_KMS
    with np.errstate(divide="ignore", invalid="ignore"):
        g_bar = Vbar2 * 1e6 / (Rad * KPC_M)             # m/s^2
        e0    = np.sqrt(a_kappa / g_bar)
        D_pt  = THREE_PI_RH_MPC * beta**2 / theta * e0**2 / (1.0 + e0)   # Mpc
    return dict(Rad=Rad, Vobs=Vobs, Vbar2=Vbar2, gas=gas_term / Rad,
                disk=disk_term / Rad, bul=bul_term / Rad, g_bar=g_bar, e0=e0, D=D_pt)

## 4. The selection criteria

The criteria are applied in order, and each galaxy is removed by the first criterion it fails. None of them uses $V_{obs}$ against a distance, and none uses $D_{SPARC}$ itself.

* **C1. Physical values.** Remove the galaxy if $V_{obs}\le0$ or $V_{bar}^2\le0$ at any point. A non-positive speed or baryonic share has no admissible projection, and $e_0$ is undefined there.
* **C2. No falling start.** Remove the galaxy if the mean $V_{obs}$ of its first three points exceeds the mean of its last three. Such profiles are dominated by unresolved central structure or truncated outer data.
* **C3. Monotonic Tone eccentricity beyond the Equivalence Point.** Find the first point with $e_0\ge1$. From there on, every next point must have a strictly larger $e_0$. A galaxy that never reaches $e_0=1$ is removed. A smooth exponential disk gives a strictly rising $e_0$ beyond the Equivalence Point, so a reversal signals rings, arms, bars or noise in the decomposition.
* **C4. Sampling of the Tone-dominated profile.** Require at least 10 points with $e_0\ge1$.
* **C5. Rotation-curve quality.** Require the SPARC quality flag $Q\le2$.
* **C6. Inclination.** Require $i\ge30^\circ$, because $V_{obs}$ is deprojected by $1/\sin i$ and $D_{RG}\propto V_{obs}^2$.

In [4]:
def first_failed_criterion(name):
    q = point_quantities(name)
    if np.any(q["Vobs"] <= 0) or np.any(q["Vbar2"] <= 0):
        return "C1"
    if q["Vobs"][:3].mean() > q["Vobs"][-3:].mean():
        return "C2"
    e0 = q["e0"]
    above = np.where(e0 >= 1.0)[0]
    if len(above) == 0:
        return "C3"
    i0 = above[0]
    if not all(e0[i] < e0[i + 1] for i in range(i0, len(e0) - 1)):
        return "C3"
    if np.sum(e0 >= 1.0) < 10:
        return "C4"
    if catalogue[name]["Q"] > 2:
        return "C5"
    if catalogue[name]["inc"] < 30.0:
        return "C6"
    return None

failed = {name: first_failed_criterion(name) for name in sorted(catalogue)}
clean  = [n for n, f in failed.items() if f is None]

CRITERIA = [("C1", "Vobs <= 0 or Vbar^2 <= 0"), ("C2", "falling start (first 3 vs last 3)"),
            ("C3", "e0 not strictly rising beyond e0 = 1"), ("C4", "fewer than 10 points with e0 >= 1"),
            ("C5", "Qual = 3"), ("C6", "i < 30 deg")]
removed_counts = {}
remaining = len(catalogue)
print(f"{'Criterion':<10}{'Content':<40}{'Removed':>8}{'Remaining':>11}")
for code_, text in CRITERIA:
    k = sum(1 for f in failed.values() if f == code_)
    removed_counts[code_] = k
    remaining -= k
    print(f"{code_:<10}{text:<40}{k:>8}{remaining:>11}")

Criterion Content                                  Removed  Remaining
C1        Vobs <= 0 or Vbar^2 <= 0                       1        174
C2        falling start (first 3 vs last 3)             18        156
C3        e0 not strictly rising beyond e0 = 1         101         55
C4        fewer than 10 points with e0 >= 1             22         33
C5        Qual = 3                                       1         32
C6        i < 30 deg                                     1         31


## 5. Surviving galaxies by origin of the catalogue distance

In [5]:
METHOD = {1: "Hubble flow", 2: "Red-giant tip", 3: "Cepheids", 4: "Ursa Major", 5: "Supernovae"}
group_counts = {}
print(f"{'Origin of D_SPARC':<20}{'In SPARC':>10}{'Clean sample':>14}")
for k, label in METHOD.items():
    n_all   = sum(1 for g in catalogue.values() if g["fD"] == k)
    n_clean = sum(1 for n in clean if catalogue[n]["fD"] == k)
    group_counts[label] = (n_all, n_clean)
    print(f"{label:<20}{n_all:>10}{n_clean:>14}")
print(f"{'Total':<20}{len(catalogue):>10}{len(clean):>14}")

Origin of D_SPARC     In SPARC  Clean sample
Hubble flow                 97            19
Red-giant tip               45             6
Cepheids                     3             1
Ursa Major                  28             5
Supernovae                   2             0
Total                      175            31


## 6. Galaxy distances and comparison targets

The distance of a galaxy is the median of the point distances over all points with $e_0\ge1$:
$$D_{RG}=\operatorname{median}\{D(\theta_i)\ :\ e_{0,i}\ge1\}.$$

The difference from a target is $(D_{RG}-D_{target})/D_{target}$, in percent. "Within $e_D$" means $|D_{RG}-D_{target}|\le e_D$, using SPARC's own quoted error.

For Hubble-flow galaxies, $D_{SPARC}$ was built with $H_0=73$. That value cancels from every data stream entering $D_{RG}$, so only the target changes. The corrected target is
$$D_{target}=D_{SPARC}\times\frac{73}{H_0},$$
and $e_D$ is rescaled by the same factor.

In [6]:
HUBBLE_FACTOR = 73.0 / H0
print(f"Hubble-flow correction factor 73/H0 = {HUBBLE_FACTOR:.5f}\n")

results = {}
for n in clean:
    q = point_quantities(n)
    use = q["e0"] >= 1.0
    results[n] = dict(points=int(use.sum()), D_RG=float(np.median(q["D"][use])),
                      D_cat=catalogue[n]["D"], eD=catalogue[n]["eD"], fD=catalogue[n]["fD"])

def diff_pct(D, T):
    return 100.0 * (D - T) / T

print("Galaxies with H0-independent catalogue distances (red-giant tip and Cepheids)")
print(f"{'Galaxy':<12}{'Method':<15}{'Points':>7}{'D_RG':>8}{'D_SPARC':>9}{'e_D':>6}{'Diff %':>9}{'Within e_D':>12}")
for n, r in results.items():
    if r["fD"] in (2, 3):
        d = diff_pct(r["D_RG"], r["D_cat"])
        print(f"{n:<12}{METHOD[r['fD']]:<15}{r['points']:>7}{r['D_RG']:>8.2f}{r['D_cat']:>9.2f}"
              f"{r['eD']:>6.2f}{d:>+9.1f}{'yes' if abs(r['D_RG']-r['D_cat'])<=r['eD'] else 'no':>12}")

print("\nUrsa Major cluster members (D_SPARC = 18.00 +/- 2.50 Mpc)")
print(f"{'Galaxy':<12}{'Points':>7}{'D_RG':>8}{'Diff %':>9}{'Within e_D':>12}")
for n, r in results.items():
    if r["fD"] == 4:
        d = diff_pct(r["D_RG"], r["D_cat"])
        print(f"{n:<12}{r['points']:>7}{r['D_RG']:>8.2f}{d:>+9.1f}{'yes' if abs(r['D_RG']-r['D_cat'])<=r['eD'] else 'no':>12}")

print("\nHubble-flow galaxies, ordered by the size of the difference from the corrected target")
print(f"{'Galaxy':<12}{'Points':>7}{'D_RG':>8}{'D_SPARC':>9}{'Diff %':>9}{'Corr.target':>12}{'Diff %':>9}{'Within e_D':>12}")
hub = [(n, r) for n, r in results.items() if r["fD"] == 1]
hub.sort(key=lambda x: abs(diff_pct(x[1]["D_RG"], x[1]["D_cat"] * HUBBLE_FACTOR)))
for n, r in hub:
    T = r["D_cat"] * HUBBLE_FACTOR
    within = abs(r["D_RG"] - T) <= r["eD"] * HUBBLE_FACTOR
    print(f"{n:<12}{r['points']:>7}{r['D_RG']:>8.2f}{r['D_cat']:>9.2f}{diff_pct(r['D_RG'], r['D_cat']):>+9.1f}"
          f"{T:>12.2f}{diff_pct(r['D_RG'], T):>+9.1f}{'yes' if within else 'no':>12}")

Hubble-flow correction factor 73/H0 = 1.07124

Galaxies with H0-independent catalogue distances (red-giant tip and Cepheids)
Galaxy      Method          Points    D_RG  D_SPARC   e_D   Diff %  Within e_D
DDO154      Red-giant tip       12    4.06     4.04  0.20     +0.5         yes
NGC2915     Red-giant tip       29    7.97     4.06  0.20    +96.4          no
NGC3741     Red-giant tip       21    4.23     3.21  0.17    +31.6          no
NGC5055     Red-giant tip       15    7.97     9.90  0.30    -19.5          no
NGC6503     Red-giant tip       27    6.90     6.26  0.31    +10.3          no
NGC6946     Red-giant tip       33    4.56     5.52  1.66    -17.4         yes
NGC7331     Cepheids            19   12.83    14.70  1.50    -12.8          no

Ursa Major cluster members (D_SPARC = 18.00 +/- 2.50 Mpc)
Galaxy       Points    D_RG   Diff %  Within e_D
NGC3769          10   16.19    -10.1         yes
NGC4100          17   18.82     +4.6         yes
NGC4157          11   15.33    -14.8 

## 7. Summary by group

For each group: the number of galaxies within 10%, 20% and 30% of the target; the median difference with its sign kept; the median of the differences with the sign ignored; and the number within $e_D$.

In [7]:
def summarise(items, corrected):
    diffs, within = [], 0
    for n, r in items:
        f = HUBBLE_FACTOR if (corrected and r["fD"] == 1) else 1.0
        T, eD = r["D_cat"] * f, r["eD"] * f
        diffs.append(diff_pct(r["D_RG"], T))
        within += abs(r["D_RG"] - T) <= eD
    a = np.abs(diffs)
    return dict(N=len(diffs), le10=int(np.sum(a <= 10)), le20=int(np.sum(a <= 20)), le30=int(np.sum(a <= 30)),
                med=float(np.median(diffs)), med_abs=float(np.median(a)), within=int(within))

groups = {
    "Red-giant tip + Cepheids":                   summarise([x for x in results.items() if x[1]["fD"] in (2, 3)], False),
    "Ursa Major":                                 summarise([x for x in results.items() if x[1]["fD"] == 4], False),
    "Hubble flow vs D_SPARC (H0=73)":             summarise([x for x in results.items() if x[1]["fD"] == 1], False),
    "Hubble flow vs corrected target":            summarise([x for x in results.items() if x[1]["fD"] == 1], True),
    "All clean galaxies (Hubble flow corrected)": summarise(list(results.items()), True),
}
print(f"{'Group':<44}{'N':>4}{'<=10%':>7}{'<=20%':>7}{'<=30%':>7}{'Med.diff':>10}{'Med|diff|':>11}{'Within e_D':>12}")
for g, s in groups.items():
    print(f"{g:<44}{s['N']:>4}{s['le10']:>7}{s['le20']:>7}{s['le30']:>7}{s['med']:>+10.1f}{s['med_abs']:>11.1f}{s['within']:>12}")

Group                                          N  <=10%  <=20%  <=30%  Med.diff  Med|diff|  Within e_D
Red-giant tip + Cepheids                       7      1      5      5      +0.5       17.4           2
Ursa Major                                     5      2      5      5      -9.3       10.1           4
Hubble flow vs D_SPARC (H0=73)                19      8     11     13      +7.0       13.0          12
Hubble flow vs corrected target               19      7     11     14      -0.1       18.8          11
All clean galaxies (Hubble flow corrected)    31     10     21     24      -0.1       15.7          17


## 8. Diagnostics of the galaxies rejected by C3

This cell examines the galaxies removed by criterion C3.

* It counts the rejected galaxies that reach $e_0=1$ with at least two points beyond it.
* For each such galaxy, it finds the largest backward step, where a step is backward when $e_{0,n+1}\le e_{0,n}$. Its size is $(e_{0,n}-e_{0,n+1})/e_{0,n}$. The cell counts the galaxies whose largest backward step exceeds 10% and those whose largest backward step is below 1%.
* It attributes each backward step to one baryonic component. The rule that reproduces the paper's numbers is this: the step is assigned to the component whose acceleration contribution $g_k=\Upsilon_k V_k^2/Rad$ increased the most between the two points.

In [8]:
n_reach, n_big, n_tiny = 0, 0, 0
steps = {"gas": 0, "disk": 0, "bulge": 0}
gal_gas, gal_disk = 0, 0
for n, f in failed.items():
    if f != "C3":
        continue
    q = point_quantities(n)
    e0 = q["e0"]
    above = np.where(e0 >= 1.0)[0]
    if len(above) == 0 or len(e0) - 1 - above[0] < 2:
        continue
    n_reach += 1
    i0 = above[0]
    back = [i for i in range(i0, len(e0) - 1) if e0[i + 1] <= e0[i]]
    largest = max((e0[i] - e0[i + 1]) / e0[i] for i in back)
    n_big  += largest > 0.10
    n_tiny += largest < 0.01
    has_gas = has_disk = False
    for i in back:
        rise = {"gas": q["gas"][i + 1] - q["gas"][i],
                "disk": q["disk"][i + 1] - q["disk"][i],
                "bulge": q["bul"][i + 1] - q["bul"][i]}
        k = max(rise, key=rise.get)
        steps[k] += 1
        has_gas  |= (k == "gas")
        has_disk |= (k == "disk")
    gal_gas += has_gas
    gal_disk += has_disk

print(f"Rejected by C3 that reach e0 = 1 with at least two points beyond it: {n_reach}")
print(f"  ... whose largest backward step exceeds 10%: {n_big}")
print(f"  ... whose largest backward step is below 1%: {n_tiny}")
print(f"Backward steps: gas-driven {steps['gas']}, disk-driven {steps['disk']}, bulge-driven {steps['bulge']}")
print(f"Galaxies with at least one gas-driven step: {gal_gas}; with at least one disk-driven step: {gal_disk}")

Rejected by C3 that reach e0 = 1 with at least two points beyond it: 98
  ... whose largest backward step exceeds 10%: 61
  ... whose largest backward step is below 1%: 8
Backward steps: gas-driven 247, disk-driven 158, bulge-driven 0
Galaxies with at least one gas-driven step: 76; with at least one disk-driven step: 68


## 9. Independence from the catalogue distance

SPARC computed `Rad` and each of $V_{gas}^2$, $V_{disk}^2$ and $V_{bul}^2$ at its assumed distance, and each is proportional to it. $V_{obs}$ is a Doppler measurement and does not depend on distance. So $\theta_i$ and $g_{bar,i}$, and therefore $e_{0,i}$ and $D(\theta_i)$, contain no trace of $D_{SPARC}$.

This cell checks it numerically. It doubles $D_{SPARC}$ together with every column proportional to it, recomputes $D_{RG}$ for all 31 clean galaxies, and prints the largest change.

In [9]:
max_shift = 0.0
for n in clean:
    q1 = point_quantities(n, scale=1.0)
    q2 = point_quantities(n, scale=2.0)
    use = q1["e0"] >= 1.0
    max_shift = max(max_shift, abs(np.median(q2["D"][use]) - np.median(q1["D"][use])))
print(f"Largest change in D_RG after doubling the catalogue distance: {max_shift:.2e} Mpc")

Largest change in D_RG after doubling the catalogue distance: 2.13e-14 Mpc


## 10. Comparison with the values printed in the paper

Every number below is copied from the paper. Each line reports PASS when the reproduced value matches it at the printed precision.

In [10]:
checks = []
def check(label, got, expected, tol=0.0):
    ok = abs(got - expected) <= tol + 1e-12
    checks.append(ok)
    print(f"{'PASS' if ok else 'FAIL'}  {label}: reproduced {got}, paper {expected}")

check("H0 [km/s/Mpc]", round(H0, 4), 68.1456)
check("a_kappa [1e-11 m/s^2]", round(a_kappa * 1e11, 4), 7.0248)
for code_, exp in zip(["C1", "C2", "C3", "C4", "C5", "C6"], [1, 18, 101, 22, 1, 1]):
    check(f"removed by {code_}", removed_counts[code_], exp)
check("clean galaxies", len(clean), 31)
for label, exp in [("Hubble flow", 19), ("Red-giant tip", 6), ("Cepheids", 1), ("Ursa Major", 5), ("Supernovae", 0)]:
    check(f"clean galaxies, {label}", group_counts[label][1], exp)

paper_D = {"DDO154": 4.06, "NGC2915": 7.97, "NGC3741": 4.23, "NGC5055": 7.97, "NGC6503": 6.90, "NGC6946": 4.56,
           "NGC7331": 12.83, "NGC3769": 16.19, "NGC4100": 18.82, "NGC4157": 15.33, "UGC06917": 20.31,
           "UGC06930": 16.33, "UGC03205": 53.48, "UGC00128": 68.71, "NGC2903": 6.99, "UGC02885": 87.56,
           "UGC08699": 42.78, "NGC6015": 18.65, "IC4202": 98.66, "UGC11455": 71.01, "UGC03580": 18.26,
           "UGC06614": 77.19, "UGC02953": 21.10, "NGC1090": 31.14, "ESO563-G021": 80.05, "NGC4559": 6.88,
           "UGC09037": 56.84, "NGC5985": 58.25, "UGC06786": 45.68, "UGC06787": 33.55, "UGC07399": 18.99}
for n, exp in paper_D.items():
    check(f"D_RG {n}", round(results[n]["D_RG"], 2), exp)

paper_summary = {
    "Red-giant tip + Cepheids":                   (7, 1, 5, 5, 0.5, 17.4, 2),
    "Ursa Major":                                 (5, 2, 5, 5, -9.3, 10.1, 4),
    "Hubble flow vs D_SPARC (H0=73)":             (19, 8, 11, 13, 7.0, 13.0, 12),
    "Hubble flow vs corrected target":            (19, 7, 11, 14, -0.1, 18.8, 11),
    "All clean galaxies (Hubble flow corrected)": (31, 10, 21, 24, -0.1, 15.7, 17)}
for g, exp in paper_summary.items():
    s = groups[g]
    got = (s["N"], s["le10"], s["le20"], s["le30"], round(s["med"], 1), round(s["med_abs"], 1), s["within"])
    for lab, a, b in zip(["N", "<=10%", "<=20%", "<=30%", "median diff", "median |diff|", "within e_D"], got, exp):
        check(f"{g}: {lab}", a, b)

check("C3 rejects reaching e0 = 1 with >= 2 points beyond", n_reach, 98)
check("  largest backward step > 10%", n_big, 61)
check("  largest backward step < 1%", n_tiny, 8)
check("gas-driven backward steps", steps["gas"], 247)
check("disk-driven backward steps", steps["disk"], 158)
check("galaxies with a gas-driven step", gal_gas, 76)
check("galaxies with a disk-driven step", gal_disk, 68)
check("D_RG change after doubling D_SPARC below 1e-9 Mpc", int(max_shift < 1e-9), 1)

print(f"\n{sum(checks)} of {len(checks)} checks passed.")

PASS  H0 [km/s/Mpc]: reproduced 68.1456, paper 68.1456
PASS  a_kappa [1e-11 m/s^2]: reproduced 7.0248, paper 7.0248
PASS  removed by C1: reproduced 1, paper 1
PASS  removed by C2: reproduced 18, paper 18
PASS  removed by C3: reproduced 101, paper 101
PASS  removed by C4: reproduced 22, paper 22
PASS  removed by C5: reproduced 1, paper 1
PASS  removed by C6: reproduced 1, paper 1
PASS  clean galaxies: reproduced 31, paper 31
PASS  clean galaxies, Hubble flow: reproduced 19, paper 19
PASS  clean galaxies, Red-giant tip: reproduced 6, paper 6
PASS  clean galaxies, Cepheids: reproduced 1, paper 1
PASS  clean galaxies, Ursa Major: reproduced 5, paper 5
PASS  clean galaxies, Supernovae: reproduced 0, paper 0
PASS  D_RG DDO154: reproduced 4.06, paper 4.06
PASS  D_RG NGC2915: reproduced 7.97, paper 7.97
PASS  D_RG NGC3741: reproduced 4.23, paper 4.23
PASS  D_RG NGC5055: reproduced 7.97, paper 7.97
PASS  D_RG NGC6503: reproduced 6.9, paper 6.9
PASS  D_RG NGC6946: reproduced 4.56, paper 4.56
PAS